## 导入模型

In [2]:
from modelscope import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# 原始模型路径和训练后的LoRA路径
base_model_path = "/home/fanzhengbo/qwen3.59b-base"  # 原始模型路径
lora_model_path = "./lora_model"                    # 你保存的LoRA模型路径

# ---------- 1. 加载分词器 ----------
tokenizer = AutoTokenizer.from_pretrained(lora_model_path)  # 直接加载你保存的分词器

# ---------- 2. 加载基础模型 ----------
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto"  # 自动分配设备（GPU/CPU）
)

# ---------- 3. 加载LoRA适配器 ----------合并
model = PeftModel.from_pretrained(base_model, lora_model_path)
model.eval()  # 切换到推理模式

`torch_dtype` is deprecated! Use `dtype` instead!
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3_5ForCausalLM(
      (model): Qwen3_5TextModel(
        (embed_tokens): Embedding(248320, 4096)
        (layers): ModuleList(
          (0-2): 3 x Qwen3_5DecoderLayer(
            (linear_attn): Qwen3_5GatedDeltaNet(
              (act): SiLUActivation()
              (conv1d): Conv1d(8192, 8192, kernel_size=(4,), stride=(1,), padding=(3,), groups=8192, bias=False)
              (norm): Qwen3_5RMSNormGated()
              (out_proj): Linear(in_features=4096, out_features=4096, bias=False)
              (in_proj_qkv): Linear(in_features=4096, out_features=8192, bias=False)
              (in_proj_z): Linear(in_features=4096, out_features=4096, bias=False)
              (in_proj_b): Linear(in_features=4096, out_features=32, bias=False)
              (in_proj_a): Linear(in_features=4096, out_features=32, bias=False)
            )
            (mlp): Qwen3_5MLP(
              (gate_proj): Linear(in_features=4096, out_features

## 保存模型的方式

In [9]:
# 兼容 PEFT 当前版本的保存方式（不再使用 save_pretrained_merged）
import torch

save_dir = "model"
save_mode = "merged_fp16"  # 可选: merged_fp16 / lora_only

# 仅保存 LoRA 适配器
if save_mode == "lora_only":
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print(f"LoRA adapter 已保存到: {save_dir}")

# 合并 LoRA 到基座模型并保存为 FP16（用于多数推理框架）
elif save_mode == "merged_fp16":
    merged_model = model.merge_and_unload()
    merged_model = merged_model.to(dtype=torch.float16)
    merged_model.save_pretrained(save_dir, safe_serialization=True)
    tokenizer.save_pretrained(save_dir)
    print(f"Merged FP16 模型已保存到: {save_dir}")



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged FP16 模型已保存到: model


In [ ]:
import os
from huggingface_hub import HfApi

# 默认走本地代理；如需覆盖可在环境变量中设置 HF_PROXY
proxy_addr = os.getenv("hf_token", "")
os.environ["http_proxy"] = proxy_addr
os.environ["https_proxy"] = proxy_addr

repo_id = "xdshuaibo/Qwen3.5-9B-lora-merged-fp16"

# 可选：如果设置了 HF_TOKEN 就使用；否则使用本机缓存登录态
token = os.getenv("hf_token")
api = HfApi(token=token)
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True, token=token)

# 推送合并后的模型和分词器
merged_model.push_to_hub(repo_id, token=token)
tokenizer.push_to_hub(repo_id, token=token)

print(f"推送成功: https://huggingface.co/{repo_id}")

In [ ]:
# GGUF / llama.cpp 格式转换
# 支持多种量化方法，如q8_0、q4_k_m、q5_k_m等

# F16（Float16）格式

# 精度类型：半精度浮点数（16位浮点数）
# 内存占用：比原始FP32（32位浮点数）减少约50%的存储空间
# 精度保留：保留了相对较高的数值精度，损失较小
# 推理性能：比FP32快，但比更低位量化格式慢
# 适用场景：当需要在内存使用和模型精度之间取得平衡时使用

# Q4_K_M格式

# 精度类型：混合4位量化格式（是GGUF量化方案的一种）
# 内存占用：比F16减少约75%的存储空间，比原始FP32减少约87.5%
# 量化策略：针对不同权重采用不同的量化策略

# 对注意力机制中的WV矩阵和前馈网络中的W2矩阵的一半使用Q6_K量化
# 对其余权重使用Q4_K量化

# 精度与速度：牺牲一定精度以获得更小的文件大小和更快的推理速度
# 适用场景：适合在资源受限设备上运行模型，如个人电脑或移动设备

# # Save to 8bit Q8_0
# if False:
#     model.save_pretrained_gguf("model", tokenizer,)
# # Remember to go to https://huggingface.co/settings/tokens for a token!
# # And change hf to your username!
# if False:
#     model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# # 保存为16位GGUF
# if False:
#     model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
# if False: # 上传到HuggingFace Hub
#     model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# # 保存为q4_k_m格式GGUF
if True:
    model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if True:# 上传到HuggingFace Hub
    model.push_to_hub_gguf("", tokenizer, quantization_method = "q4_k_m", token = "")

# # 保存多种GGUF格式（批量导出更高效）
# if False:
#     model.push_to_hub_gguf(
#         "hf/model", # Change hf to your username!
#         tokenizer,
#         quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
#         token = "", # Get a token at https://huggingface.co/settings/tokens
#     )